# 🔍 NVIDIA Nemotron-3-Nano-30B Diagnostic Inference Notebook

This notebook is designed to run inference using the baseline **NVIDIA Nemotron-3-Nano-30B-A3B-BF16** model. It generates both the internal reasoning thoughts (wrapped in `<think>` tags) and the final response, parses them, and compares them to the ground-truth answers in the dataset.

### 📋 Goals:
1. **Load model & tokenizer** efficiently for inference (`use_cache=True`).
2. **Apply the native chat template** to ensure the model's `<think>` tags are fully utilized.
3. **Parse thoughts vs. final answers** cleanly.
4. **Compare against ground truth** to output a diagnostic CSV file of correct/incorrect outputs.

## 🛠️ Step 1: Environment Configuration

In [ ]:
import os
import shutil
import stat
import glob
import site
import sys

# --- 1. CRITICAL NVIDIA / TRITON PTXAS PERMISSION PATCH ---
print("Configuring Triton binary overrides...")
candidates = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*script/triton/backends/nvidia/bin/ptxas-blackwell")
if candidates:
    src = candidates[0]
    dst_dir = "/tmp/triton-bin"
    os.makedirs(dst_dir, exist_ok=True)
    dst = f"{dst_dir}/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    os.environ["PATH"] = f"{dst_dir}:" + os.environ["PATH"]
    os.environ["CUDA_MODULE_LOADING"] = "LAZY"
    os.environ["TRITON_DISABLE_AUTOTUNE"] = "1"
    os.environ["USE_TRITON"] = "0"
    os.environ["PTXAS_PATH"] = dst
    print(f"PTXAS override successful: {dst}")
else:
    print("ptxas-blackwell not found in utility scripts. Assuming other architecture or local run.")

# --- 2. CRITICAL CACHING DIRECTORY REDIRECTS ---
print("Configuring caching directories to /tmp...")
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TORCH_EXTENSIONS_DIR"] = "/tmp/torch_extensions"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"

# --- 3. NVIDIA CUTLASS PATH CONFIGURATION ---
search_path = "/kaggle/usr/lib/notebooks/*/nvidia*utility*script/nvidia_cutlass_dsl/python_packages/"
candidates = glob.glob(search_path)
if candidates:
    cutlass_pkg_path = candidates[0]
    site.addsitedir(cutlass_pkg_path)
    print(f"NVIDIA Cutlass DSL package path loaded from: {cutlass_pkg_path}")

## 📥 Step 2: Import Dependencies

In [ ]:
import torch
import re
import polars as pl
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import kagglehub

print(f"CUDA Available: {torch.cuda.is_available()}")

## 🤖 Step 3: Load Model & Tokenizer

In [ ]:
# --- CONFIGURATION ---
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
ADAPTER_PATH = None  # Set to a local path (e.g. "/kaggle/working" or adapter folder) to load trained weights

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model for inference (using bfloat16 and auto device mapping)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    use_cache=True  # Important: use_cache=True speeds up generation significantly
)

if ADAPTER_PATH is not None and os.path.exists(ADAPTER_PATH):
    print(f"Loading trained LoRA adapter weights from {ADAPTER_PATH}...")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print("Model loaded successfully.")

## 🛠️ Step 4: Define Output Parsing & Answer Extraction Helpers

In [ ]:
def extract_boxed_content(text):
    """
    Extracts the content inside \boxed{...} LaTeX blocks.
    Prioritizes the last matching boxed structure, handling basic nested brackets.
    """
    # Find all matches of \boxed{...}
    matches = re.findall(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}", text)
    if matches:
        return matches[-1].strip()  # Return the last (final) boxed answer if multiple
    
    # Fallback heuristics: if no \boxed{}, find the last numeric value or the last non-empty line
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    if lines:
        last_line = lines[-1]
        # Look for numbers at the end
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", last_line)
        if numbers:
            return numbers[-1]
        return last_line
    return ""

def parse_model_output(output_text):
    """
    Parses the generated response to separate thinking process and the final answer.
    """
    thought = ""
    answer_raw = output_text
    
    # Check for <think> and </think> tags
    if "<think>" in output_text:
        parts = output_text.split("<think>", 1)
        if "</think>" in parts[1]:
            thought_parts = parts[1].split("</think>", 1)
            thought = thought_parts[0].strip()
            answer_raw = thought_parts[1].strip()
        else:
            thought = parts[1].strip()
            answer_raw = ""
            
    extracted_answer = extract_boxed_content(answer_raw)
    return thought, answer_raw, extracted_answer

## 🧪 Step 5: Test a Single Prompt

In [ ]:
def test_single_prompt(prompt_text):
    # Formulate a structured chat message prompt asking the model to reason and wrap in \boxed{}
    messages = [
        {"role": "user", "content": prompt_text + "\nReason step-by-step and place your final answer inside \\boxed{}."}
    ]
    
    # Apply standard chat template
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.0,  # Greedier decoding for deterministic reasoning
            top_p=1.0,
            do_sample=False
        )
        
    # Extract only the generated part
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    thought, raw_response, final_answer = parse_model_output(generated_text)
    
    print("="*80)
    print(f"PROMPT:\n{prompt_text}\n")
    print(f"THOUGHT PROCESS:\n{thought}\n")
    print(f"RAW RESPONSE:\n{raw_response}\n")
    print(f"EXTRACTED ANSWER: {final_answer}")
    print("="*80)

# Run a test with a simple Roman numeral prompt
test_single_prompt("Convert the number 42 into Roman numerals.")

## 📊 Step 6: Dataset Diagnostic Inference Loop

This step loops through a subset of the dataset prompts, runs model inference, compares model outputs to the ground-truth answers, and logs them for diagnosis.

In [ ]:
# --- DATA CONFIGURATION ---
data_path = "DATA/train.csv" if os.path.exists("DATA/train.csv") else "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pl.read_csv(data_path)

# Limit to a small sample for fast evaluation and diagnosis (e.g. 30 samples)
DIAGNOSTIC_SAMPLES = 30
df_sample = df.head(DIAGNOSTIC_SAMPLES)

results = []

print(f"Running diagnostic inference on {len(df_sample)} examples...")

for row in tqdm(df_sample.iter_rows(named=True), total=len(df_sample)):
    prompt_text = row["prompt"]
    ground_truth = row["answer"]
    
    formatted_prompt = prompt_text + "\nReason step-by-step and write your final answer inside \\boxed{}."
    
    messages = [{"role": "user", "content": formatted_prompt}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.0,
            top_p=1.0,
            do_sample=False
        )
    
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    thought, raw_response, final_answer = parse_model_output(generated_text)
    
    # Compare prediction to ground truth (exact string strip match)
    is_correct = (final_answer.strip() == ground_truth.strip())
    
    results.append({
        "id": row.get("id", ""),
        "prompt": prompt_text,
        "ground_truth": ground_truth,
        "model_thought": thought,
        "model_raw_response": raw_response,
        "model_answer": final_answer,
        "is_correct": is_correct
    })

# Convert results to Polars DataFrame
df_results = pl.DataFrame(results)

# Calculate and print accuracy
accuracy = df_results["is_correct"].mean()
print(f"\nDiagnostic Accuracy on {DIAGNOSTIC_SAMPLES} samples: {accuracy * 100:.2f}%")

# Save diagnostic outputs for downstream analysis
output_csv = "diagnostic_results.csv"
df_results.write_csv(output_csv)
print(f"Diagnostic results successfully saved to: {output_csv}")

## 🛑 Step 7: View and Diagnose Failure Cases

In [ ]:
df_failures = df_results.filter(pl.col("is_correct") == False)
print(f"Total failure cases: {len(df_failures)}\n")

# Display first 5 failures to diagnose where the model's logic or formatting failed
for i in range(min(5, len(df_failures))):
    row = df_failures.row(i, named=True)
    print(f"❌ FAILURE CASE {i+1} (ID: {row['id']})")
    print(f"Prompt: {row['prompt']}")
    print(f"Ground Truth: {row['ground_truth']}")
    print(f"Model Extracted Answer: {row['model_answer']}")
    print(f"Model Raw Response: {row['model_raw_response']}")
    print(f"Model Thought Process:\n{row['model_thought']}")
    print("-"*80)